In [1]:
import polars as pl

In [2]:
data_jan_2009 = pl.read_parquet("C:/Users/ekadw/Documents/DATA/NY_Taxi/2009/yellow_taxi/yellow_tripdata_2009-01.parquet", 
                                columns=['Trip_Pickup_DateTime','Trip_Dropoff_DateTime','Passenger_Count','Trip_Distance',
                                         'Payment_Type','Fare_Amt','Tip_Amt'])

In [3]:
len(data_jan_2009)

14092413

In [4]:
data_jan_2009.head()

Trip_Pickup_DateTime,Trip_Dropoff_DateTime,Passenger_Count,Trip_Distance,Payment_Type,Fare_Amt,Tip_Amt
str,str,i64,f64,str,f64,f64
"""2009-01-04 02:52:00""","""2009-01-04 03:02:00""",1,2.63,"""CASH""",8.9,0.0
"""2009-01-04 03:31:00""","""2009-01-04 03:38:00""",3,4.55,"""Credit""",12.1,2.0
"""2009-01-03 15:43:00""","""2009-01-03 15:57:00""",5,10.35,"""Credit""",23.7,4.74
"""2009-01-01 20:52:58""","""2009-01-01 21:14:00""",1,5.0,"""CREDIT""",14.9,3.05
"""2009-01-24 16:18:23""","""2009-01-24 16:24:56""",1,0.4,"""CASH""",3.7,0.0


In [5]:
data_jan_2009.columns

['Trip_Pickup_DateTime',
 'Trip_Dropoff_DateTime',
 'Passenger_Count',
 'Trip_Distance',
 'Payment_Type',
 'Fare_Amt',
 'Tip_Amt']

In [6]:
data_jan_2009 = data_jan_2009.rename({
    "Trip_Pickup_DateTime": "trip_pickup_dateTime",
    "Trip_Dropoff_DateTime": "trip_dropoff_dateTime",
    "Passenger_Count": "passenger_count",
    "Trip_Distance": "trip_distance",
    "Payment_Type": "payment_type",
    "Fare_Amt": "fare_amt",
    "Tip_Amt": "tip_amt"
})

In [10]:
data_jan_2009 = data_jan_2009.filter(
    (pl.col("passenger_count") >= 0) & (pl.col("trip_distance") >= 0) & (pl.col("trip_distance") <= 50) & 
    (pl.col("fare_amt") >= 0) & (pl.col("tip_amt") >= 0)
)
len(data_jan_2009)

14092413

In [11]:
mapping = {
    "Credit": 0,
    "CREDIT": 0,
    "CASH": 1,
    "Cash": 1,
    "No Charge": 2,
    "Dispute": 3
}

data_jan_2009 = data_jan_2009.with_columns(
    pl.col("payment_type").replace(mapping)
)

In [12]:
data_jan_2009.head()

trip_pickup_dateTime,trip_dropoff_dateTime,passenger_count,trip_distance,payment_type,fare_amt,tip_amt
str,str,i64,f64,str,f64,f64
"""2009-01-04 02:52:00""","""2009-01-04 03:02:00""",1,2.63,"""1""",8.9,0.0
"""2009-01-04 03:31:00""","""2009-01-04 03:38:00""",3,4.55,"""0""",12.1,2.0
"""2009-01-03 15:43:00""","""2009-01-03 15:57:00""",5,10.35,"""0""",23.7,4.74
"""2009-01-01 20:52:58""","""2009-01-01 21:14:00""",1,5.0,"""0""",14.9,3.05
"""2009-01-24 16:18:23""","""2009-01-24 16:24:56""",1,0.4,"""1""",3.7,0.0


In [14]:
data_jan_2009 = data_jan_2009.with_columns([
    pl.col("trip_pickup_dateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
    pl.col("trip_dropoff_dateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
    pl.col("payment_type").cast(pl.Int64)
])

In [16]:
data_jan_2009.head()

trip_pickup_dateTime,trip_dropoff_dateTime,passenger_count,trip_distance,payment_type,fare_amt,tip_amt
datetime[μs],datetime[μs],i64,f64,i64,f64,f64
2009-01-04 02:52:00,2009-01-04 03:02:00,1,2.63,1,8.9,0.0
2009-01-04 03:31:00,2009-01-04 03:38:00,3,4.55,0,12.1,2.0
2009-01-03 15:43:00,2009-01-03 15:57:00,5,10.35,0,23.7,4.74
2009-01-01 20:52:58,2009-01-01 21:14:00,1,5.0,0,14.9,3.05
2009-01-24 16:18:23,2009-01-24 16:24:56,1,0.4,1,3.7,0.0


In [ ]:
data_jan_2009 = data_jan_2009.with_columns(
    (pl.col("trip_dropoff_dateTime") - pl.col("trip_pickup_dateTime")).alias("duration_days")
)

In [ ]:
['Trip_Pickup_DateTime','Trip_Dropoff_DateTime','Passenger_Count','Trip_Distance','Payment_Type','Fare_Amt','Tip_Amt']

In [1]:
import polars as pl

df = pl.DataFrame({
    "category": ["CASH", "Credit", "No Charge", "CREDIT", "Cash", "Dispute"]
})

mapping = {
    "CASH": 1,
    "Cash": 1,
    "Credit": 0,
    "CREDIT": 0,
    "No Charge": 2,
    "Dispute": 3
}

df = df.with_columns(
    pl.col("category").replace(mapping).alias("category_num")
)
print(df)

shape: (6, 2)
┌───────────┬──────────────┐
│ category  ┆ category_num │
│ ---       ┆ ---          │
│ str       ┆ str          │
╞═══════════╪══════════════╡
│ CASH      ┆ 1            │
│ Credit    ┆ 0            │
│ No Charge ┆ 2            │
│ CREDIT    ┆ 0            │
│ Cash      ┆ 1            │
│ Dispute   ┆ 3            │
└───────────┴──────────────┘
